<a href="https://colab.research.google.com/github/abdelhalimyasser/Olympics-Data-Pipeline-AI-Classification/blob/main/notebooks/classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

In [ ]:
# load & explore data
sports_summry_df = pd.read_csv('sports_summry.csv')
sports_summry_df.head()

,season,discipline,number_of_olympics,number_of_gold_medals,number_of_silver_medals,number_of_bronze_medals,total_medals,athlete_with_most_medals,athlete_with_most_gold_medals
0,Summer,Archery,18,76,74,66,216,Hubert van Innis,Hubert van Innis
1,Summer,Rowing,29,282,282,286,850,Elisabeta Lipă,Elisabeta Lipă
2,Summer,Rugby,7,10,11,7,28,Theresa Fitzpatrick,Theresa Fitzpatrick
3,Summer,Sailing,29,205,197,190,592,Ben Ainslie,Ben Ainslie
4,Summer,Shooting,28,302,303,301,906,Carl Osburn,Carl Osburn


In [ ]:
# load & explore data
discontinued_sports_df = pd.read_csv('discontinued_sports.csv')
discontinued_sports_df.head()

,season,discipline,contest_year,number_of_olympics,gold_medals,silver_medals,bronze_medals
0,Summer,Basque pelota,1900,1,1,0,0
1,Summer,Breaking,2024,1,2,2,2
2,Summer,Croquet,1900,1,3,2,2
3,Summer,Jeu de paume,1908,1,1,1,1
4,Summer,Karate,2020,1,8,8,16


In [ ]:
# fit & prepare the df
sports_summry_df = sports_summry_df.drop(columns=['athlete_with_most_medals',	'athlete_with_most_gold_medals'])
sports_summry_df['is_discontinued'] = 0
sports_summry_df.head()

,season,discipline,number_of_olympics,number_of_gold_medals,number_of_silver_medals,number_of_bronze_medals,total_medals,is_discontinued
0,Summer,Archery,18,76,74,66,216,0
1,Summer,Rowing,29,282,282,286,850,0
2,Summer,Rugby,7,10,11,7,28,0
3,Summer,Sailing,29,205,197,190,592,0
4,Summer,Shooting,28,302,303,301,906,0


In [ ]:
# fit & prepare the df
discontinued_sports_df = discontinued_sports_df.drop(columns=['contest_year'])
discontinued_sports_df = discontinued_sports_df.rename(columns={'gold_medals': 'number_of_gold_medals',	'silver_medals' : 'number_of_silver_medals',	'bronze_medals': 'number_of_bronze_medals'})
discontinued_sports_df['total_medals'] = discontinued_sports_df['number_of_gold_medals'] + discontinued_sports_df['number_of_silver_medals'] + discontinued_sports_df['number_of_bronze_medals']
discontinued_sports_df['is_discontinued'] = 1
discontinued_sports_df.head()

,season,discipline,number_of_olympics,number_of_gold_medals,number_of_silver_medals,number_of_bronze_medals,total_medals,is_discontinued
0,Summer,Basque pelota,1,1,0,0,1,1
1,Summer,Breaking,1,2,2,2,6,1
2,Summer,Croquet,1,3,2,2,7,1
3,Summer,Jeu de paume,1,1,1,1,3,1
4,Summer,Karate,1,8,8,16,32,1


In [ ]:
# concate the two tables
df = pd.concat([sports_summry_df, discontinued_sports_df], ignore_index=True)
df = df.drop(columns=['discipline'])
df.head()

,season,number_of_olympics,number_of_gold_medals,number_of_silver_medals,number_of_bronze_medals,total_medals,is_discontinued
0,Summer,18,76,74,66,216,0
1,Summer,29,282,282,286,850,0
2,Summer,7,10,11,7,28,0
3,Summer,29,205,197,190,592,0
4,Summer,28,302,303,301,906,0


In [ ]:
# mapping the season
season_mapping = {'Summer': 1, 'Winter': 0}
df['season'] = df['season'].map(season_mapping)

In [ ]:
# set the X, y features
features = ['season', 'number_of_olympics', 'number_of_gold_medals', 'number_of_silver_medals', 'number_of_bronze_medals', 'total_medals']
X = df[features]
y = df['is_discontinued']

In [ ]:
# train & test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# my function to evaluate the models
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score, r2_score, mean_absolute_error, mean_squared_error

def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    is_classification = hasattr(model, "predict_proba") or len(np.unique(y_test)) < 20

    print(f"--- Testing Report: {type(model).__name__} ---")

    if is_classification:
        print("\n[Classification Metrics]")
        print(f"Accuracy:  {accuracy_score(y_test, predictions):.4f}")
        print(f"F1 Score:  {f1_score(y_test, predictions, average='weighted'):.4f}")
        print("\nDetailed Report:")
        print(classification_report(y_test, predictions))
    else:
        print("\n[Regression Metrics]")
        print(f"R2 Score: {r2_score(y_test, predictions):.4f}")
        print(f"MAE:      {mean_absolute_error(y_test, predictions):.4f}")
        print(f"RMSE:     {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")

    print("\n--- Checking 10 Random Data Predictions ---")
    indices = np.random.choice(X_test.index if isinstance(X_test, pd.DataFrame) else len(X_test), 10, replace=False)

    for i, idx in enumerate(indices, 1):
        sample_x = X_test.iloc[[i]] if hasattr(X_test, 'iloc') else X_test[idx].reshape(1, -1)
        actual_y = y_test.iloc[i] if hasattr(y_test, 'iloc') else y_test[idx]

        pred_y = model.predict(sample_x)[0]

        print(f"Sample {i} | Actual: {actual_y} | Predicted: {pred_y} | {'✅ Match' if actual_y == pred_y else '❌ Diff'}")

    print("\n----------------------------------------------------------\n")

In [ ]:
# classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(),
    "Naive Bayes": GaussianNB()
}

In [ ]:
# make the classification and evaluate it
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    evaluate_model(model, X_train, X_test, y_train, y_test)

--- Testing Report: LogisticRegression ---

[Classification Metrics]
Accuracy:  0.9286
F1 Score:  0.9230

Detailed Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        11
           1       1.00      0.67      0.80         3

    accuracy                           0.93        14
   macro avg       0.96      0.83      0.88        14
weighted avg       0.93      0.93      0.92        14


--- Checking 10 Random Data Predictions ---
Sample 1 | Actual: 0 | Predicted: 0 | ✅ Match
Sample 2 | Actual: 0 | Predicted: 0 | ✅ Match
Sample 3 | Actual: 0 | Predicted: 0 | ✅ Match
Sample 4 | Actual: 0 | Predicted: 0 | ✅ Match
Sample 5 | Actual: 1 | Predicted: 1 | ✅ Match
Sample 6 | Actual: 0 | Predicted: 0 | ✅ Match
Sample 7 | Actual: 0 | Predicted: 0 | ✅ Match
Sample 8 | Actual: 1 | Predicted: 1 | ✅ Match
Sample 9 | Actual: 0 | Predicted: 0 | ✅ Match
Sample 10 | Actual: 0 | Predicted: 0 | ✅ Match

-------------------------------------------

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# save the models to use it
os.makedirs('models', exist_ok=True)

joblib.dump(models["Logistic Regression"], 'models/logistic_regression.pkl')
joblib.dump(models["KNN"], 'models/knn.pkl')
joblib.dump(models["Decision Tree"], 'models/decision_tree.pkl')
joblib.dump(models["Random Forest"], 'models/random_forest.pkl')
joblib.dump(models["SVM"], 'models/svm.pkl')
joblib.dump(models["Naive Bayes"], 'models/naive_bayes.pkl')

['models/naive_bayes.pkl']